# PromptShield: data cleaning

Export **exact** and **normalized** training variants with common validation/test. Labels: **0 = benign, 1 = injection**. Original files stay unchanged. Exact retains original text; normalized lowercases all prompts.


## 1. Setup

Requires pandas (`%pip install pandas` if needed). Run from the repository root or the `cleaning/` folder.


In [ ]:
import json
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_DIR = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data" / "train.json").is_file()
)
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "cleaning" / "data"
SPLITS = ("train", "validation", "test")
PRIORITY = ("test", "validation", "train")
LABELS = {0: "benign", 1: "injection"}
REMOVE_CROSS_SPLIT_EXACT = True

assert set(PRIORITY) == set(SPLITS) and len(PRIORITY) == len(SPLITS)
assert OUTPUT_DIR.resolve() != DATA_DIR.resolve(), "Choose a separate output folder."


## 2. Load data

Read the three published JSON splits from `data/`.


In [ ]:
raw = {}
input_metadata = {}
for split in SPLITS:
    path = DATA_DIR / f"{split}.json"
    content = path.read_bytes()
    rows = json.loads(content.decode("utf-8"))
    if not isinstance(rows, list):
        raise ValueError(f"{path}: expected a JSON list")
    raw[split] = rows
    input_metadata[split] = {
        "file": str(path), "rows": len(rows),
    }
display(pd.DataFrame(input_metadata).T[["file", "rows"]])


## 3. Validate rows

Remove invalid/blank prompts and labels other than integer 0 or 1. Record original row positions and removal reasons.


In [ ]:
def invalid_reason(row):
    if not isinstance(row, dict):
        return "record_not_object"
    if "prompt" not in row or row["prompt"] is None:
        return "missing_prompt"
    if not isinstance(row["prompt"], str):
        return "prompt_not_string"
    if not row["prompt"].strip():
        return "blank_prompt"
    if "label" not in row or row["label"] is None:
        return "missing_label"
    if type(row["label"]) is not int or row["label"] not in LABELS:
        return "invalid_label"
    return None

removed = []
valid = {}
schema_report = []
for split, rows in raw.items():
    records = []
    extra_fields = Counter()
    for source_row, row in enumerate(rows):
        if isinstance(row, dict):
            extra_fields.update(set(row) - {"prompt", "label"})
        reason = invalid_reason(row)
        if reason:
            removed.append({"split": split, "source_row": source_row, "reason": reason})
        else:
            records.append({"source_row": source_row, "prompt": row["prompt"], "label": row["label"]})
    valid[split] = pd.DataFrame(records, columns=["source_row", "prompt", "label"])
    schema_report.append({"split": split, "valid_rows": len(records), "extra_fields": dict(extra_fields)})
display(pd.DataFrame(schema_report))
display(pd.DataFrame(Counter(row["reason"] for row in removed).items(), columns=["reason", "rows"]))


## 4. Duplicates and overlap

`exact`: identical text. `normalized`: ignore case and repeated whitespace; keep punctuation. Conflicts mean the same prompt has both labels.


In [ ]:
def normalize_for_audit(text):
    return " ".join(text.casefold().split())

def comparison_keys(frame, mode):
    return frame["prompt"] if mode == "exact" else frame["prompt"].map(normalize_for_audit)

def duplicate_audit(frames):
    results = []
    for split, frame in frames.items():
        for mode in ("exact", "normalized"):
            keys = comparison_keys(frame, mode)
            groups = pd.DataFrame({"key": keys, "label": frame["label"]}).groupby("key")["label"].nunique()
            results.append({"split": split, "comparison": mode,
                            "extra_duplicate_rows": int(keys.duplicated().sum()),
                            "conflicting_label_groups": int(groups.gt(1).sum())})
    return pd.DataFrame(results)

def overlap_audit(frames):
    results = []
    for left, right in combinations(SPLITS, 2):
        for mode in ("exact", "normalized"):
            left_keys = comparison_keys(frames[left], mode)
            right_keys = comparison_keys(frames[right], mode)
            shared = set(left_keys) & set(right_keys)
            labels = defaultdict(set)
            for split, keys in ((left, left_keys), (right, right_keys)):
                for key, label in zip(keys, frames[split]["label"]):
                    if key in shared:
                        labels[key].add(label)
            results.append({"left": left, "right": right, "comparison": mode,
                            "shared_unique_prompts": len(shared),
                            "right_rows_seen_in_left": int(right_keys.isin(shared).sum()),
                            "conflicting_label_groups": sum(len(values) > 1 for values in labels.values())})
    return pd.DataFrame(results)

before_duplicates = duplicate_audit(valid)
before_overlap = overlap_audit(valid)
display(before_duplicates)
display(before_overlap)


### Prompt examples

Up to 3 duplicate/conflict groups per split, with a 300-character preview. Adjust the settings below to show more.


In [ ]:
N_EXAMPLES = 3  # Per category and split.
PREVIEW_CHARS = 300
SAMPLE_MODE = "exact"  # Or "normalized".
assert SAMPLE_MODE in ("exact", "normalized")

example_rows = []
for split, frame in valid.items():
    groups = frame.assign(key=comparison_keys(frame, SAMPLE_MODE)).groupby("key", sort=False)
    shown = {"duplicate": 0, "conflict": 0}
    for _, group in groups:
        if len(group) < 2:
            continue
        kind = "conflict" if group["label"].nunique() > 1 else "duplicate"
        if shown[kind] >= N_EXAMPLES:
            continue
        text = group.iloc[0]["prompt"]
        example_rows.append({
            "split": split, "type": kind,
            "prompt": text[:PREVIEW_CHARS] + ("?" if len(text) > PREVIEW_CHARS else ""),
            "occurrences": len(group), "labels": sorted(group["label"].unique().tolist()),
            "source_rows": group["source_row"].head(6).tolist(),
        })
        shown[kind] += 1

if example_rows:
    with pd.option_context("display.max_colwidth", PREVIEW_CHARS + 1):
        display(pd.DataFrame(example_rows))
else:
    print("No duplicate or conflicting prompts found.")
if not any(row["type"] == "conflict" for row in example_rows):
    print("No conflicting labels found.")


## 5. Two training variants

**Exact:** remove exact conflicts/duplicates and overlap (test > validation > train).
**Normalized:** additionally remove normalized train conflicts/duplicates and train overlap with validation/test. Both variants use the same evaluation rows/labels. Normalized lowercases all retained prompts (train/validation/test); exact preserves original text.


In [ ]:
def record_removals(split, frame, reason, kept_in_split=None):
    for source_row in frame["source_row"]:
        record = {"split": split, "source_row": int(source_row), "reason": reason}
        if kept_in_split is not None:
            record["kept_in_split"] = kept_in_split
        removed.append(record)

cleaned = {}
for split, frame in valid.items():
    conflict_keys = frame.groupby("prompt")["label"].nunique()
    conflict_mask = frame["prompt"].isin(conflict_keys[conflict_keys.gt(1)].index)
    record_removals(split, frame.loc[conflict_mask], "within_split_label_conflict")
    frame = frame.loc[~conflict_mask].copy()
    duplicate_mask = frame.duplicated("prompt", keep="first")
    record_removals(split, frame.loc[duplicate_mask], "exact_duplicate_within_split")
    cleaned[split] = frame.loc[~duplicate_mask].copy()

if REMOVE_CROSS_SPLIT_EXACT:
    seen = set()
    owners = {}
    for split in PRIORITY:
        frame = cleaned[split]
        overlap_mask = frame["prompt"].isin(seen)
        for owner, group in frame.loc[overlap_mask].groupby(frame.loc[overlap_mask, "prompt"].map(owners)):
            record_removals(split, group, "exact_overlap_between_splits", kept_in_split=owner)
        cleaned[split] = frame.loc[~overlap_mask].copy()
        for prompt in cleaned[split]["prompt"]:
            owners[prompt] = split
        seen.update(cleaned[split]["prompt"])


# Independent normalized training copy, with common evaluation splits.
variants = {"exact": cleaned}
variant_removals = {"exact": [dict(row) for row in removed]}
normalized = {split: frame.copy() for split, frame in cleaned.items()}
normalized_removed = [dict(row) for row in removed]

def remove_normalized_rows(frame, mask, reason):
    normalized_removed.extend(
        {"split": "train", "source_row": int(row), "reason": reason}
        for row in frame.loc[mask, "source_row"]
    )
    return frame.loc[~mask].copy()

train = normalized["train"]
keys = comparison_keys(train, "normalized")
conflicts = train.groupby(keys)["label"].nunique()
train = remove_normalized_rows(train, keys.isin(conflicts[conflicts.gt(1)].index),
                               "normalized_label_conflict")
keys = comparison_keys(train, "normalized")
train = remove_normalized_rows(train, keys.duplicated(), "normalized_duplicate_within_train")
evaluation_keys = set().union(*(
    set(comparison_keys(normalized[split], "normalized"))
    for split in ("validation", "test")
))
train = remove_normalized_rows(
    train, comparison_keys(train, "normalized").isin(evaluation_keys),
    "normalized_train_overlap_with_evaluation",
)
normalized["train"] = train
for frame in normalized.values():
    frame["prompt"] = frame["prompt"].map(str.lower)
variants["normalized"] = normalized
variant_removals["normalized"] = normalized_removed


## 6. Compare counts

Evaluation rows/labels are identical; normalized text is lowercase in every split.


In [ ]:
summary_rows = []
for variant, frames in variants.items():
    for split, frame in frames.items():
        summary_rows.append({
            "variant": variant, "split": split,
            "rows": len(frame), "removed": len(raw[split]) - len(frame),
            "benign": int(frame["label"].eq(0).sum()),
            "injection": int(frame["label"].eq(1).sum()),
        })
summary = pd.DataFrame(summary_rows)
display(summary.set_index(["variant", "split"]))


## 7. Verify

Check row accounting, expected text transformation, unchanged labels, and deduplication.


In [ ]:
for variant, frames in variants.items():
    for split, frame in frames.items():
        assert not frame.empty, f"{variant}/{split}: no usable rows"
        mode = "normalized" if variant == "normalized" and split == "train" else "exact"
        # Keep evaluation rows fixed, even when lowercase makes texts coincide.
        comparison_frame = frame
        if variant == "normalized" and split != "train":
            comparison_frame = frame.assign(prompt=[raw[split][int(i)]["prompt"] for i in frame["source_row"]])
        assert not comparison_keys(comparison_frame, mode).duplicated().any()
        retained_ids = set(frame["source_row"])
        removed_ids = [row["source_row"] for row in variant_removals[variant] if row["split"] == split]
        assert len(removed_ids) == len(set(removed_ids))
        assert retained_ids.isdisjoint(removed_ids)
        assert retained_ids | set(removed_ids) == set(range(len(raw[split])))
        for row in frame.to_dict("records"):
            original = raw[split][row["source_row"]]
            expected_text = original["prompt"].lower() if variant == "normalized" else original["prompt"]
            assert row["prompt"] == expected_text
            assert type(row["label"]) is int and row["label"] == original["label"]
            assert invalid_reason(row) is None
    if REMOVE_CROSS_SPLIT_EXACT:
        for left, right in combinations(SPLITS, 2):
            left_original = {raw[left][int(i)]["prompt"] for i in frames[left]["source_row"]}
            right_original = {raw[right][int(i)]["prompt"] for i in frames[right]["source_row"]}
            assert left_original.isdisjoint(right_original)
for split in ("validation", "test"):
    exact_eval = variants["exact"][split]
    normalized_eval = variants["normalized"][split]
    assert exact_eval[["source_row", "label"]].equals(normalized_eval[["source_row", "label"]])
    assert normalized_eval["prompt"].equals(exact_eval["prompt"].map(str.lower))
assert set(comparison_keys(variants["normalized"]["train"], "normalized")).isdisjoint(evaluation_keys)
print("Checks passed: same evaluation rows/labels; normalized text is lowercase.")


## 8. Export

Save only the three JSON splits per variant. Counts and checks stay in the notebook.


In [ ]:
input_paths = {(DATA_DIR / f"{split}.json").resolve() for split in SPLITS}
for variant, frames in variants.items():
    folder = OUTPUT_DIR / variant
    assert all((folder / f"{split}.json").resolve() not in input_paths for split in SPLITS)
    folder.mkdir(parents=True, exist_ok=True)
    for split, frame in frames.items():
        records = frame[["prompt", "label"]].to_dict("records")
        path = folder / f"{split}.json"
        path.write_text(json.dumps(records, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        assert json.loads(path.read_text(encoding="utf-8")) == records
    print(f"Saved: {folder} (train, validation, test)")


## Training comparison

Use `DATA_DIR = Path("cleaning/data/exact")` or `Path("cleaning/data/normalized")` in the analysis notebook. Retrain from scratch with the same model/settings/seed and compare on the same validation rows using each variant's preprocessing (original vs lowercase). Keep test for the final evaluation.
